# Skew & Salting

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

Two claims under test, each checked directly against real evidence rather than assumed:
1. `groupBy(key).agg(F.collect_list("amount"))` on a dataset where one key covers ~60% of rows produces one straggler task on the reduce-side (post-shuffle) stage, with visibly larger shuffle-read bytes (and corroborating duration) than the rest -- `collect_list` carries every row's value across the shuffle instead of pre-combining a key's rows into one row per mapper (the way `count()`/`sum()` do), so the hot key's full row volume genuinely lands on one reduce task -- read directly from per-task REST data, not eyeballed off the UI.
2. Salting the hot key with a random `0-9` suffix, aggregating on the salted key, then stripping the suffix and re-aggregating (merging the sub-key arrays back with `F.flatten`) produces the *same* final per-key values (verified against the un-salted result, not just assumed) while the salted straggler's own shuffle-read-bytes load drops by roughly `NUM_SALT_BUCKETS`-fold (>=3x, hard-asserted below) versus the un-salted straggler -- it does not flatten the whole `spark.sql.shuffle.partitions=200`-task distribution, since only 10 of 200 tasks ever carry hot-key mass.

No `tools/datagen/` utility exists in this repo (same gap `content/aqe/notebook.ipynb`'s own note flags) -- this notebook generates the skewed dataset inline instead, the same way the AQE topic does, with one hot key deliberately covering ~60% of rows to match this topic's acceptance criteria (`docs/requirements/curriculum-topics-2026-07.md` US-C2).

In [ ]:
import json
import statistics
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("skew-salting").getOrCreate()
app_id = spark.sparkContext.applicationId
print("Spark version:", spark.version)
print("Application id:", app_id)


def _get(path):
    url = f"http://localhost:4040/api/v1/applications/{app_id}/{path}"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


def stages_snapshot():
    """Current `/api/v1/applications/<id>/stages` list -- same helper/
    disposition as dag-lazy-evaluation's jobs_snapshot() and
    window-functions'/caching-persistence's own stages/storage snapshot
    helpers: used here to isolate the *new* stage(s) a groupBy shuffle
    produces, not to read task-level detail directly (that's
    reduce_side_task_metrics() below)."""
    return _get("stages")


def reduce_side_task_metrics(before_stage_ids):
    """Per-task `duration` (ms) and shuffle-read bytes for the reduce-side
    (post-shuffle) stage of the groupBy just run -- the real evidence for
    'is there one straggler task', not assumed from the mockup's numbers.

    Picks the new stage with `shuffleReadBytes > 0` and the most tasks: a
    groupBy(key).count()/agg() plan has a map-side (partial-aggregate,
    reads input, no shuffle read) stage and a reduce-side (post-shuffle)
    stage -- only the reduce-side stage is fed by the Exchange this topic's
    skew actually lands on (same disposition as window-functions' own
    shuffle_read_stages selection, which flags the same map-side/reduce-side
    ambiguity for the identical reason)."""
    new_stages = [s for s in stages_snapshot() if s["stageId"] not in before_stage_ids]
    shuffle_read_stages = [s for s in new_stages if s.get("shuffleReadBytes", 0) > 0]
    stage = max(shuffle_read_stages, key=lambda s: s["numTasks"])
    tasks = _get(f"stages/{stage['stageId']}/{stage.get('attemptId', 0)}/taskList?length=1000")
    metrics = []
    for t in tasks:
        if not isinstance(t, dict):
            continue
        shuffle_read = (t.get("taskMetrics") or {}).get("shuffleReadMetrics") or {}
        shuffle_read_bytes = (shuffle_read.get("localBytesRead", 0) or 0) + (
            shuffle_read.get("remoteBytesRead", 0) or 0
        )
        metrics.append({
            "index": t.get("index"),
            "duration": t.get("duration"),
            "shuffleReadBytes": shuffle_read_bytes,
        })
    return stage, metrics

## 1. Build a dataset with one key covering ~60% of rows

`HOT_KEY` gets ~60% of all rows; the remaining 40% spread evenly across `NUM_NORMAL_KEYS` normal keys -- deliberate, verified skew (same ~60%-hot-key shape the AQE topic's own dataset uses, but a single hot key here rather than three, and no join -- just a `groupBy`, the case AQE's skew-join split cannot reach).

In [ ]:
FACT_ROWS = 2_000_000
NUM_NORMAL_KEYS = 500
HOT_KEY = "hot-0"
NUM_INPUT_PARTITIONS = 24

def make_row(i: int) -> Row:
    # Deterministic ~60% hot-key skew, no numpy dependency -- matches this
    # topic's inline-generation scope (see the intro cell's note).
    if i % 5 < 3:
        key = HOT_KEY
    else:
        key = f"key-{i % NUM_NORMAL_KEYS}"
    return Row(key=key, amount=float(i % 997))

fact_rdd = spark.sparkContext.parallelize(range(FACT_ROWS), NUM_INPUT_PARTITIONS).map(make_row)
skewed_df = spark.createDataFrame(fact_rdd)

# Confirm the skew actually landed before relying on it below.
top_counts = skewed_df.groupBy("key").count().orderBy(F.desc("count")).limit(5)
top_counts.show()
hot_key_rows = top_counts.filter(F.col("key") == HOT_KEY).first()["count"]
hot_key_fraction = hot_key_rows / FACT_ROWS
print(f"{HOT_KEY} share of rows: {hot_key_fraction:.0%}")
assert hot_key_fraction > 0.5, f"expected {HOT_KEY} to dominate with >50% of rows, got {hot_key_fraction:.0%}"

## 2. `groupBy(key).agg(collect_list(...))` with no salting -- hypothesize first

**Hypothesis:** on the reduce-side (post-shuffle) stage, do you expect every task's shuffle-read bytes (and duration) to be roughly even, or do you expect one task -- the one `HOT_KEY` hashes to -- to be visibly larger than the rest? Unlike `count()`, `collect_list` can't pre-combine a key's rows into a single partial row on each mapper before the shuffle -- the whole point of a list is to retain every element -- so `HOT_KEY`'s rows should genuinely all cross the shuffle boundary onto whichever task it hashes to.

In [ ]:
before_stage_ids = {s["stageId"] for s in stages_snapshot()}
counts_no_salt = skewed_df.groupBy("key").agg(F.sort_array(F.collect_list("amount")).alias("vals"))
# .foreach(), NOT .collect(): HOT_KEY's collected list has ~1.2M elements --
# collecting it to the driver would risk OOM/hang. sort_array() makes the
# later correctness comparison order-independent (collect_list's element
# order isn't deterministic across runs).
counts_no_salt.foreach(lambda row: None)

no_salt_stage, no_salt_metrics = reduce_side_task_metrics(before_stage_ids)
no_salt_durations = [m["duration"] for m in no_salt_metrics]
no_salt_shuffle_bytes = [m["shuffleReadBytes"] for m in no_salt_metrics]

straggler = max(no_salt_metrics, key=lambda m: m["duration"])
median_duration = statistics.median(no_salt_durations)
median_shuffle_bytes = statistics.median(no_salt_shuffle_bytes)

print(f"Stage {no_salt_stage['stageId']}: {len(no_salt_metrics)} tasks")
print(f"Median task duration: {median_duration:.0f}ms  Straggler task {straggler['index']}: {straggler['duration']}ms")
print(f"Median task shuffle-read bytes: {median_shuffle_bytes:.0f}  Straggler: {straggler['shuffleReadBytes']}")

# collect_list is a TypedImperativeAggregate (ObjectHashAggregateExec): its
# partial-aggregation buffer IS the accumulated array, so each mapper's
# partial output for HOT_KEY still carries all of that mapper's hot-key
# values across the shuffle -- unlike count()/sum(), this genuinely reflects
# row-volume skew, so this is now a reliable hard assertion (see
# docs/architecture/skew-salting-demo-mechanism.md).
assert straggler["shuffleReadBytes"] >= 2 * median_shuffle_bytes, (
    f"expected the straggler task's shuffle-read bytes ({straggler['shuffleReadBytes']}) to be "
    f">= 2x the median ({median_shuffle_bytes:.0f}) -- collect_list carries every row's value across "
    "the shuffle, so HOT_KEY's ~60% of rows should land almost entirely on one reduce task"
)
print(f"Straggler/median shuffle-read-bytes ratio: {straggler['shuffleReadBytes'] / median_shuffle_bytes:.2f}x")

checkpoint(counts_no_salt, topic="skew-salting")
print("Checkpoint written -- click 'Reveal self-check' on the topic page and check the stage-metrics "
      "panel's per-task duration quantiles for this stage before moving on.")

## 3. Salt the hot key, aggregate, then strip the suffix and re-aggregate

**Hypothesis:** after salting `HOT_KEY` into 10 sub-keys (`hot-0_0` ... `hot-0_9`) and aggregating on the salted key, do you expect the reduce-side shuffle-read-bytes spread to flatten out, or just move the imbalance somewhere else? Verify the final per-key values still match the un-salted result exactly before trusting the comparison.

In [ ]:
NUM_SALT_BUCKETS = 10

salted_df = skewed_df.withColumn(
    "salted_key",
    F.when(
        F.col("key") == HOT_KEY,
        F.concat(F.col("key"), F.lit("_"), (F.rand() * NUM_SALT_BUCKETS).cast("int")),
    ).otherwise(F.col("key")),
)

before_stage_ids = {s["stageId"] for s in stages_snapshot()}
salted_counts = salted_df.groupBy("salted_key").agg(F.sort_array(F.collect_list("amount")).alias("vals"))
salted_counts.foreach(lambda row: None)  # same OOM-avoidance reasoning as step 2

salted_stage, salted_metrics = reduce_side_task_metrics(before_stage_ids)
salted_durations = [m["duration"] for m in salted_metrics]
salted_shuffle_bytes = [m["shuffleReadBytes"] for m in salted_metrics]

salted_straggler = max(salted_metrics, key=lambda m: m["duration"])
salted_median_duration = statistics.median(salted_durations)
salted_median_shuffle_bytes = statistics.median(salted_shuffle_bytes)
salted_max_shuffle_bytes = max(salted_shuffle_bytes)

print(f"Stage {salted_stage['stageId']}: {len(salted_metrics)} tasks")
print(f"Median task duration: {salted_median_duration:.0f}ms  Slowest task {salted_straggler['index']}: {salted_straggler['duration']}ms")
print(f"Median task shuffle-read bytes: {salted_median_shuffle_bytes:.0f}  Max: {salted_max_shuffle_bytes}")

# ponytail: with only NUM_SALT_BUCKETS=10 atomic sub-keys hash-partitioned
# into 200 fixed shuffle partitions, at most 10 of those 200 tasks carry any
# hot-key mass -- the other ~190 stay cold, so the ALL-tasks median stays
# near the cold baseline and salted_max/median does NOT collapse to ~1x
# (it's still ~20-30x here). Salting at N=10 doesn't flatten the global
# distribution; it cuts the STRAGGLER's load by ~N-fold (one task goes from
# 100% of the hot key's rows to ~1/10) -- that's the real, measured effect,
# so assert that reduction instead of a flatness claim the physics can't
# deliver at this bucket count. True flattening would need N in the
# thousands (buckets >> partitions, so the law of large numbers spreads
# hot mass over every partition) -- pedagogically absurd for a "10 sub-keys"
# lesson. See docs/architecture/skew-salting-demo-mechanism.md, "Salted-side
# assert" section, for the full derivation and the sweep that found this.
assert salted_max_shuffle_bytes < straggler["shuffleReadBytes"] / 3, (
    f"expected salting into {NUM_SALT_BUCKETS} buckets to cut the straggler's shuffle-read bytes "
    f">=3x (un-salted straggler {straggler['shuffleReadBytes']}, salted max {salted_max_shuffle_bytes}, "
    f"achieved {straggler['shuffleReadBytes'] / salted_max_shuffle_bytes:.1f}x)"
)
print(f"Un-salted straggler / salted max shuffle-read-bytes reduction: "
      f"{straggler['shuffleReadBytes'] / salted_max_shuffle_bytes:.1f}x")
print(f"Max/median shuffle-read-bytes ratio after salting: {salted_max_shuffle_bytes / salted_median_shuffle_bytes:.2f}x "
      "(stays well above 1x at N=10 buckets -- salting cuts the straggler's own load, it does not "
      "flatten the whole distribution; see the note above)")

# Strip the "_N" suffix off the hot key's sub-keys and re-aggregate -- a
# second, much smaller shuffle (one row per original key, not per original
# row) that merges the salted sub-keys' arrays back into the true per-key
# result. F.flatten (not sum): the payload is now a per-key array, not a
# count, so re-aggregation concatenates arrays rather than adding numbers.
stripped_df = salted_counts.withColumn(
    "key",
    F.when(F.col("salted_key").startswith(HOT_KEY + "_"), F.lit(HOT_KEY)).otherwise(F.col("salted_key")),
)
final_counts = stripped_df.groupBy("key").agg(F.sort_array(F.flatten(F.collect_list("vals"))).alias("vals"))

# Correctness check: the salted-then-stripped result must exactly match the
# un-salted groupBy from step 2 -- salting must not change the answer, only
# how it's computed. Both sides are sort_array()ed so equal multisets compare
# equal despite collect_list's non-deterministic element order. This stays a
# distributed filter+count, never a .collect() of HOT_KEY's large array.
mismatches = (
    final_counts.withColumnRenamed("vals", "salted_vals")
    .join(counts_no_salt.withColumnRenamed("vals", "no_salt_vals"), "key")
    .filter(F.col("salted_vals") != F.col("no_salt_vals"))
    .count()
)
print(f"Mismatches between salted-and-stripped values and the un-salted values: {mismatches}")
assert mismatches == 0, "expected salting to produce the identical per-key values as the un-salted groupBy"

checkpoint(final_counts, topic="skew-salting")
print("Checkpoint written -- Reveal again after the comparison below.")

## 4. Compare the per-task shuffle-read-bytes reduction before and after salting

The straggler-load-reduction contrast in shuffle-read bytes (with duration as corroborating evidence) is the real, measured evidence for what salting did -- not just the plan text (a skewed and a salted `groupBy` compile to the same `HashAggregate` -> `Exchange` -> `HashAggregate` plan shape either way). Note: with `NUM_SALT_BUCKETS=10` sub-keys spread across `spark.sql.shuffle.partitions=200` reduce tasks, salting does *not* flatten the whole 200-task distribution to one even level -- it cuts the *straggler's own load* by roughly `NUM_SALT_BUCKETS`-fold, which is the measurable, hard-asserted claim below.

In [ ]:
no_salt_bytes_reduction = straggler["shuffleReadBytes"] / salted_max_shuffle_bytes
no_salt_ratio = straggler["duration"] / median_duration
salted_ratio = salted_straggler["duration"] / salted_median_duration

print(f"Straggler-load reduction from salting (shuffle-read bytes): {no_salt_bytes_reduction:.1f}x "
      f"(un-salted straggler {straggler['shuffleReadBytes']} -> salted max {salted_max_shuffle_bytes})")
print(f"Un-salted:  straggler/median duration ratio = {no_salt_ratio:.2f}x  (straggler {straggler['duration']}ms vs median {median_duration:.0f}ms)")
print(f"Salted:     slowest/median duration ratio    = {salted_ratio:.2f}x  (slowest {salted_straggler['duration']}ms vs median {salted_median_duration:.0f}ms)")
print(f"All salted task durations (ms): {sorted(salted_durations)}")

# Duration is corroborating evidence here, not the hard check -- ms-scale
# task timing stays noisy even though shuffle-read bytes (the hard checks in
# the two cells above -- un-salted straggler >=2x median, and salting
# cutting that straggler's load >=3x) is a deterministic, reliable signal.
if not (salted_ratio < no_salt_ratio):
    print("NOTE: salted duration ratio did not drop below the un-salted ratio this run "
          f"({salted_ratio:.2f}x vs {no_salt_ratio:.2f}x) -- shuffle-read bytes (the hard checks above) "
          "is the reliable signal; duration stays noisy at this job's millisecond scale.")